# 04 · Model Evaluation
Comprehensive evaluation of trained risk models using held-out test data.

**Metrics covered:**
- ROC curve and AUC
- Precision-Recall curve
- Confusion matrix
- Calibration curve (reliability diagram)
- Threshold sensitivity analysis

In [ ]:
import sys, warnings, json
from pathlib import Path
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report, brier_score_loss,
    ConfusionMatrixDisplay,
)
from sklearn.calibration import calibration_curve

from src.ml.data.diabetes_preprocessor import DiabetesPreprocessor
from src.ml.data.hypertension_preprocessor import HypertensionPreprocessor

sns.set_theme(style='darkgrid', palette='muted')
MODELS_DIR = Path('../models/saved')
DATA_DIR   = '../data/raw/2017_2018'
print('Ready.')

## 1 · Prepare test sets

In [ ]:
def load_model(prefix):
    files = sorted(MODELS_DIR.glob(f'{prefix}_model_*.joblib'))
    return joblib.load(files[-1]) if files else None

# Diabetes
X_d, y_d = DiabetesPreprocessor().load_and_preprocess(DATA_DIR)
_, X_d_test, _, y_d_test = train_test_split(X_d, y_d, test_size=0.2, stratify=y_d, random_state=42)
diab_model = load_model('diabetes')

# Hypertension
X_h, y_h = HypertensionPreprocessor().load_and_preprocess(DATA_DIR)
_, X_h_test, _, y_h_test = train_test_split(X_h, y_h, test_size=0.2, stratify=y_h, random_state=42)
htn_model = load_model('hypertension')

print(f'Diabetes test:     {len(y_d_test):,} samples  ({y_d_test.mean():.1%} positive)')
print(f'Hypertension test: {len(y_h_test):,} samples  ({y_h_test.mean():.1%} positive)')

## 2 · ROC curves — both models

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for name, model, X_test, y_test, color in [
    ('Diabetes',     diab_model, X_d_test, y_d_test, 'steelblue'),
    ('Hypertension', htn_model,  X_h_test, y_h_test, 'coral'),
]:
    if model is None:
        print(f'{name} model not found — skipping')
        continue
    probs = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={roc_auc:.4f})')

ax.plot([0,1],[0,1], 'k--', lw=1, label='Random (AUC=0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

## 3 · Precision-Recall curves

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for name, model, X_test, y_test, color in [
    ('Diabetes',     diab_model, X_d_test, y_d_test, 'steelblue'),
    ('Hypertension', htn_model,  X_h_test, y_h_test, 'coral'),
]:
    if model is None: continue
    probs = model.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, probs)
    ap = average_precision_score(y_test, probs)
    ax.plot(rec, prec, color=color, lw=2, label=f'{name} (AP={ap:.4f})')
    baseline = y_test.mean()
    ax.axhline(baseline, color=color, ls=':', lw=1, alpha=0.5)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves')
ax.legend()
plt.tight_layout()
plt.show()

## 4 · Confusion matrices at 0.5 threshold

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, (name, model, X_test, y_test) in zip(axes, [
    ('Diabetes',     diab_model, X_d_test, y_d_test),
    ('Hypertension', htn_model,  X_h_test, y_h_test),
]):
    if model is None:
        ax.text(0.5, 0.5, f'{name}\nnot found', ha='center', va='center')
        continue
    preds = model.predict(X_test)
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Negative','Positive'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name} Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 5 · Calibration (reliability diagram)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (name, model, X_test, y_test) in zip(axes, [
    ('Diabetes',     diab_model, X_d_test, y_d_test),
    ('Hypertension', htn_model,  X_h_test, y_h_test),
]):
    if model is None:
        ax.text(0.5, 0.5, 'Model not found', ha='center', va='center')
        continue
    probs = model.predict_proba(X_test)[:, 1]
    frac_pos, mean_pred = calibration_curve(y_test, probs, n_bins=8)
    brier = brier_score_loss(y_test, probs)
    ax.plot([0,1],[0,1], 'k--', lw=1, label='Perfect calibration')
    ax.plot(mean_pred, frac_pos, 's-', color='steelblue', lw=2, label=f'Model (Brier={brier:.4f})')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_title(f'{name} — Calibration curve')
    ax.legend()
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

## 6 · Threshold sensitivity — F1 and precision/recall trade-off

In [ ]:
if diab_model is not None:
    probs = diab_model.predict_proba(X_d_test)[:, 1]
    thresholds = np.linspace(0.05, 0.95, 50)

    results = []
    for t in thresholds:
        preds = (probs >= t).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_d_test, preds).ravel()
        prec = tp / (tp + fp + 1e-9)
        rec  = tp / (tp + fn + 1e-9)
        f1   = 2 * prec * rec / (prec + rec + 1e-9)
        results.append({'threshold': t, 'precision': prec, 'recall': rec, 'f1': f1})

    res_df = pd.DataFrame(results)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(res_df['threshold'], res_df['precision'], label='Precision', color='steelblue')
    ax.plot(res_df['threshold'], res_df['recall'],    label='Recall',    color='coral')
    ax.plot(res_df['threshold'], res_df['f1'],        label='F1',        color='green', lw=2)
    best_t = res_df.loc[res_df['f1'].idxmax(), 'threshold']
    ax.axvline(best_t, ls='--', color='gray', lw=1, label=f'Best F1 @ {best_t:.2f}')
    ax.axvline(0.5,    ls=':',  color='black', lw=1, label='Default threshold 0.5')
    ax.set_xlabel('Decision threshold')
    ax.set_ylabel('Score')
    ax.set_title('Diabetes — Threshold sensitivity (test set)')
    ax.legend()
    ax.set_ylim([0, 1.05])
    plt.tight_layout()
    plt.show()
    print(f'Best F1 threshold: {best_t:.2f}  (F1 = {res_df["f1"].max():.4f})')

## 7 · Evaluation summary table

In [ ]:
rows = []
for name, model, X_test, y_test in [
    ('Diabetes',     diab_model, X_d_test, y_d_test),
    ('Hypertension', htn_model,  X_h_test, y_h_test),
]:
    if model is None: continue
    probs = model.predict_proba(X_test)[:, 1]
    preds = model.predict(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    rows.append({
        'Model':      name,
        'Accuracy':   (tp+tn) / len(y_test),
        'ROC-AUC':    auc(*roc_curve(y_test, probs)[:2]),
        'Sensitivity': tp / (tp + fn),
        'Specificity': tn / (tn + fp),
        'Precision':  tp / (tp + fp),
        'F1':         2*tp / (2*tp + fp + fn),
        'Brier':      brier_score_loss(y_test, probs),
    })

pd.DataFrame(rows).set_index('Model').style.format('{:.4f}').background_gradient(cmap='Blues')

## 8 · Interpretation notes

- **Diabetes model** achieves ~94.9% accuracy and 0.967 AUC — excellent discrimination.
  High precision (0.997) means very few false positives; moderate recall (0.727) means
  some diabetes cases are missed — acceptable for a screening tool where the cost of
  false positives (unnecessary anxiety) may be higher than false negatives.
- **Hypertension model** has lower AUC because blood pressure readings are excluded.
  The model predicts *risk* from demographic and lifestyle factors, not from BP itself.
- Both models show good calibration (Brier scores near 0.04-0.08).

➡️ Continue to `05_shap_analysis.ipynb`